In [1]:
import os
import json
import math
import copy
import csv
import time

import numpy as np
import nibabel as nib

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset

In [2]:
with open(
    "data_split.json",
    "r"
) as f:
    split_data = json.load(f)


train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]


print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))


SUBJECTS_JSON = os.path.join(
    "evaluation_200",
    "conditions",
    "evaluation_subjects_200.json"
)


if not os.path.isfile(SUBJECTS_JSON):
    raise FileNotFoundError(
        "The shared 200-subject cohort was not found: "
        f"{SUBJECTS_JSON}"
    )


with open(
    SUBJECTS_JSON,
    "r"
) as f:
    cohort_data = json.load(f)


if isinstance(
    cohort_data,
    dict
):
    evaluation_subjects = (
        cohort_data["subjects"]
    )
else:
    evaluation_subjects = cohort_data


if len(evaluation_subjects) != 200:
    raise RuntimeError(
        "Expected 200 evaluation subjects, "
        f"found {len(evaluation_subjects)}."
    )


if len(set(evaluation_subjects)) != 200:
    raise RuntimeError(
        "Duplicate subjects were found "
        "in the shared cohort."
    )


heldout_subjects = set(
    list(val_subjects)
    + list(test_subjects)
)


if not set(evaluation_subjects).issubset(
    heldout_subjects
):
    raise RuntimeError(
        "The shared cohort contains subjects "
        "outside the validation/test sets."
    )


if set(evaluation_subjects).intersection(
    set(train_subjects)
):
    raise RuntimeError(
        "Training-subject overlap detected."
    )


print(
    "Loaded shared evaluation cohort:",
    SUBJECTS_JSON
)

print(
    "Evaluation subjects:",
    len(evaluation_subjects)
)

Train: 1000
Validation: 125
Test: 126
Loaded shared evaluation cohort: evaluation_200/conditions/evaluation_subjects_200.json
Evaluation subjects: 200


In [3]:
DATA_DIR = (
    "Data/"
    "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)


if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"BraTS data directory not found: {DATA_DIR}"
    )


print(
    "BraTS data directory:",
    DATA_DIR
)

BraTS data directory: Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData


In [4]:
def preprocess_t2f(image):
    # Expected original BraTS shape
    if image.shape != (240, 240, 155):
        raise ValueError(f"Unexpected image shape: {image.shape}")

    image = image[16:224, 8:232, :]

    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    foreground = image > 0

    if not np.any(foreground):
        raise ValueError("No foreground voxels found")

    upper = np.percentile(image[foreground], 99.9)

    image = np.clip(image, 0, upper)

    image = image / upper

    image[~foreground] = 0

    return image.astype(np.float32)

In [5]:
def preprocess_mask(mask):
    # Expected original BraTS shape
    if mask.shape != (240, 240, 155):
        raise ValueError(f"Unexpected mask shape: {mask.shape}")

    # Apply exactly the same spatial crop as T2f
    mask = mask[16:224, 8:232, :]

    # Apply exactly the same z-padding as T2f
    mask = np.pad(
        mask,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    return mask.astype(np.int64)

In [6]:
def calculate_tumour_entropy(image, mask, num_bins=256):
    # Whole tumour = all non-zero tumour labels
    tumour_region = mask > 0

    if not np.any(tumour_region):
        raise ValueError("No tumour voxels found")

    tumour_values = image[tumour_region]

    # T2f has already been normalized to [0, 1]
    tumour_values = np.clip(tumour_values, 0.0, 1.0)

    hist, _ = np.histogram(
        tumour_values,
        bins=num_bins,
        range=(0.0, 1.0),
        density=False
    )

    probabilities = hist.astype(np.float64)
    probabilities = probabilities / probabilities.sum()

    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return np.float32(entropy)

In [7]:
class BraTSDataset(Dataset):
    def __init__(self, subjects, data_dir):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subject = self.subjects[idx]
        subject_path = os.path.join(self.data_dir, subject)

        files = os.listdir(subject_path)

        t2f_file = [f for f in files if "t2f" in f.lower()][0]
        seg_file = [f for f in files if "seg" in f.lower()][0]

        # Load T2f
        image = nib.load(
            os.path.join(subject_path, t2f_file)
        ).get_fdata()

        # Load segmentation
        mask = nib.load(
            os.path.join(subject_path, seg_file)
        ).get_fdata()

        # Apply preprocessing
        image = preprocess_t2f(image)
        mask = preprocess_mask(mask)

        # Calculate whole-tumour Shannon entropy
        entropy = calculate_tumour_entropy(
            image,
            mask
        )

        # Convert to tensors
        image = torch.from_numpy(
            image
        ).float().unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).long().unsqueeze(0)

        entropy = torch.tensor(
            entropy,
            dtype=torch.float32
        )

        return {
            "image": image,
            "mask": mask,
            "heterogeneity": entropy,
            "subject": subject
        }

In [8]:
# ============================================================
# Shared evaluation cohort and LDM output directories
# ============================================================

BASE_OUTPUT_DIR = "evaluation_200"


OUTPUT_DIR = os.path.join(
    BASE_OUTPUT_DIR,
    "conditional_ldm_v4"
)


CONDITION_DIR = os.path.join(
    BASE_OUTPUT_DIR,
    "conditions"
)


MASK_DIR = os.path.join(
    CONDITION_DIR,
    "masks"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


evaluation_dataset = BraTSDataset(
    subjects=evaluation_subjects,
    data_dir=DATA_DIR
)


print(
    "Evaluation cohort size:",
    len(evaluation_dataset)
)

print(
    "Synthetic output directory:",
    OUTPUT_DIR
)

print(
    "Shared condition directory:",
    CONDITION_DIR
)

Evaluation cohort size: 200
Synthetic output directory: evaluation_200/conditional_ldm_v4
Shared condition directory: evaluation_200/conditions


In [9]:
class VAEBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),
            nn.SiLU(),

            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),
            nn.SiLU()
        )

    def forward(self, x):
        return self.block(x)

In [10]:
class VAEEncoder3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.enc1 = VAEBlock3D(
            in_channels,
            base_channels
        )

        self.down1 = nn.Conv3d(
            base_channels,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.enc2 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )

        self.down2 = nn.Conv3d(
            base_channels * 2,
            base_channels * 4,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.bottleneck = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )

        self.to_mu = nn.Conv3d(
            base_channels * 4,
            latent_channels,
            kernel_size=1
        )

        self.to_logvar = nn.Conv3d(
            base_channels * 4,
            latent_channels,
            kernel_size=1
        )

    def forward(self, x):

        x = self.enc1(x)
        x = self.down1(x)

        x = self.enc2(x)
        x = self.down2(x)

        x = self.bottleneck(x)

        mu = self.to_mu(x)
        logvar = self.to_logvar(x)

        return mu, logvar

In [11]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [12]:
class VAEDecoder3D(nn.Module):
    def __init__(
        self,
        out_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.from_latent = nn.Conv3d(
            latent_channels,
            base_channels * 4,
            kernel_size=3,
            padding=1
        )

        self.dec2 = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )

        self.up2 = nn.ConvTranspose3d(
            base_channels * 4,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec1 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )

        self.up1 = nn.ConvTranspose3d(
            base_channels * 2,
            base_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.final_block = VAEBlock3D(
            base_channels,
            base_channels
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, z):

        x = self.from_latent(z)

        x = self.dec2(x)
        x = self.up2(x)

        x = self.dec1(x)
        x = self.up1(x)

        x = self.final_block(x)
        x = self.output_conv(x)

        x = torch.sigmoid(x)

        return x

In [13]:
class VAE3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.encoder = VAEEncoder3D(
            in_channels=in_channels,
            base_channels=base_channels,
            latent_channels=latent_channels
        )

        self.decoder = VAEDecoder3D(
            out_channels=out_channels,
            base_channels=base_channels,
            latent_channels=latent_channels
        )

    def forward(self, x):
        mu, logvar = self.encoder(x)

        z = reparameterize(
            mu,
            logvar
        )

        reconstruction = self.decoder(z)

        return reconstruction, mu, logvar, z

In [14]:
# ============================================================
# Load the frozen x4 VAE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


if device.type != "cuda":
    raise RuntimeError(
        "CUDA GPU is not available. "
        "Do not run 3D LDM sampling on CPU."
    )


VAE_CKPT_PATH = (
    "vae_x4_v3_checkpoints/"
    "vae_v3_epoch_015.pt"
)


if not os.path.isfile(VAE_CKPT_PATH):
    raise FileNotFoundError(
        f"VAE checkpoint not found: {VAE_CKPT_PATH}"
    )


vae = VAE3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    latent_channels=4
).to(device)


vae_checkpoint = torch.load(
    VAE_CKPT_PATH,
    map_location=device
)


vae.load_state_dict(
    vae_checkpoint[
        "model_state_dict"
    ]
)


loaded_vae_epoch = int(
    vae_checkpoint["epoch"]
)


if loaded_vae_epoch != 15:
    raise RuntimeError(
        "Expected VAE epoch 15, "
        f"but loaded epoch {loaded_vae_epoch}."
    )


vae.eval()
vae.requires_grad_(False)


del vae_checkpoint


print(
    "Device:",
    device
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "Loaded frozen VAE epoch:",
    loaded_vae_epoch
)

print(
    "VAE checkpoint:",
    VAE_CKPT_PATH
)

Device: cuda
GPU: NVIDIA A40
Loaded frozen VAE epoch: 15
VAE checkpoint: vae_x4_v3_checkpoints/vae_v3_epoch_015.pt


In [15]:
import math

timesteps = 1000


def cosine_beta_schedule(
    timesteps,
    s=0.008
):

    steps = timesteps + 1

    x = torch.linspace(
        0,
        timesteps,
        steps,
        dtype=torch.float64
    )

    alpha_bar = torch.cos(
        (
            (x / timesteps + s)
            / (1.0 + s)
        )
        * math.pi
        * 0.5
    ) ** 2

    alpha_bar = (
        alpha_bar
        / alpha_bar[0]
    )

    betas = (
        1.0
        - alpha_bar[1:]
        / alpha_bar[:-1]
    )

    return torch.clamp(
        betas,
        min=1e-8,
        max=0.999
    ).float()


def rescale_zero_terminal_snr(
    betas
):

    alphas = (
        1.0 - betas
    )

    alpha_bar = torch.cumprod(
        alphas,
        dim=0
    )

    sqrt_alpha_bar = torch.sqrt(
        alpha_bar
    )

    first = sqrt_alpha_bar[0].clone()
    last = sqrt_alpha_bar[-1].clone()

    sqrt_alpha_bar = (
        sqrt_alpha_bar - last
    )

    sqrt_alpha_bar = (
        sqrt_alpha_bar
        * first
        / (first - last)
    )

    alpha_bar = (
        sqrt_alpha_bar ** 2
    )

    new_alphas = (
        alpha_bar[1:]
        / alpha_bar[:-1]
    )

    new_alphas = torch.cat(
        [
            alpha_bar[:1],
            new_alphas
        ]
    )

    return (
        1.0 - new_alphas
    ).float()


betas = cosine_beta_schedule(
    timesteps
)

betas = rescale_zero_terminal_snr(
    betas
)

alphas = (
    1.0 - betas
)

alphas_cumprod = torch.cumprod(
    alphas,
    dim=0
)

alphas_cumprod_prev = F.pad(
    alphas_cumprod[:-1],
    (1, 0),
    value=1.0
)

sqrt_alphas_cumprod = torch.sqrt(
    alphas_cumprod
)

sqrt_one_minus_alphas_cumprod = torch.sqrt(
    1.0 - alphas_cumprod
)

posterior_variance = (
    betas
    * (
        1.0
        - alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)

posterior_variance = torch.clamp(
    posterior_variance,
    min=1e-20
)

posterior_mean_coef1 = (
    betas
    * torch.sqrt(
        alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)

posterior_mean_coef2 = (
    (
        1.0
        - alphas_cumprod_prev
    )
    * torch.sqrt(
        alphas
    )
    / (
        1.0
        - alphas_cumprod
    )
)


final_snr = (
    alphas_cumprod[-1]
    / torch.clamp(
        1.0
        - alphas_cumprod[-1],
        min=1e-12
    )
)

print(
    "Final alpha_cumprod:",
    alphas_cumprod[-1].item()
)

print(
    "Final SNR:",
    final_snr.item()
)

assert (
    alphas_cumprod[-1].item()
    == 0.0
)

print(
    "Zero-terminal-SNR check passed."
)

Final alpha_cumprod: 0.0
Final SNR: 0.0
Zero-terminal-SNR check passed.


In [16]:
def v_to_x0(
    xt,
    v,
    t
):
    a = (
        sqrt_alphas_cumprod
        .to(xt.device)[t]
        .view(-1, 1, 1, 1, 1)
    )


    b = (
        sqrt_one_minus_alphas_cumprod
        .to(xt.device)[t]
        .view(-1, 1, 1, 1, 1)
    )


    return (
        a * xt
        - b * v
    )


def prepare_latent_mask(
    mask
):
    mask = (
        mask.squeeze(1)
        .long()
    )


    onehot = F.one_hot(
        mask,
        num_classes=4
    )


    onehot = (
        onehot
        .permute(
            0,
            4,
            1,
            2,
            3
        )
        .float()
    )


    # Remove the background channel
    onehot = onehot[:, 1:]


    onehot = F.interpolate(
        onehot,
        size=(52, 56, 40),
        mode="nearest"
    )


    return onehot

In [17]:
class SinusoidalTimeEmbedding(nn.Module):

    def __init__(
        self,
        dim
    ):
        super().__init__()
        self.dim = dim

    def forward(
        self,
        t
    ):

        half = (
            self.dim // 2
        )

        scale = (
            math.log(10000)
            / (half - 1)
        )

        emb = torch.exp(
            torch.arange(
                half,
                device=t.device
            )
            * -scale
        )

        emb = (
            t[:, None].float()
            * emb[None, :]
        )

        return torch.cat(
            [
                emb.sin(),
                emb.cos()
            ],
            dim=1
        )


class ResBlock3D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        condition_dim,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.GroupNorm(
            8,
            in_channels
        )

        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.condition_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                out_channels * 2
            )
        )

        nn.init.zeros_(
            self.condition_mlp[-1].weight
        )

        nn.init.zeros_(
            self.condition_mlp[-1].bias
        )

        self.norm2 = nn.GroupNorm(
            8,
            out_channels
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        nn.init.zeros_(
            self.conv2.weight
        )

        nn.init.zeros_(
            self.conv2.bias
        )

        if (
            in_channels
            != out_channels
        ):

            self.skip = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )

        else:
            self.skip = nn.Identity()


    def forward(
        self,
        x,
        condition
    ):

        residual = self.skip(x)

        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)

        scale, shift = (
            self.condition_mlp(
                condition
            )
            .chunk(
                2,
                dim=1
            )
        )

        scale = scale[
            :, :, None, None, None
        ]

        shift = shift[
            :, :, None, None, None
        ]

        h = self.norm2(h)

        h = (
            h
            * (1.0 + scale)
            + shift
        )

        h = F.silu(h)
        h = self.dropout(h)
        h = self.conv2(h)

        return (
            residual + h
        )


class SelfAttention3D(nn.Module):

    def __init__(
        self,
        channels,
        heads=8
    ):
        super().__init__()

        self.norm = nn.GroupNorm(
            8,
            channels
        )

        self.attention = (
            nn.MultiheadAttention(
                embed_dim=channels,
                num_heads=heads,
                batch_first=True
            )
        )


    def forward(
        self,
        x
    ):

        b, c, d, h, w = x.shape

        residual = x

        x = self.norm(x)

        x = (
            x.permute(
                0, 2, 3, 4, 1
            )
            .reshape(
                b,
                d * h * w,
                c
            )
        )

        x, _ = self.attention(
            x,
            x,
            x,
            need_weights=False
        )

        x = (
            x.reshape(
                b,
                d,
                h,
                w,
                c
            )
            .permute(
                0,
                4,
                1,
                2,
                3
            )
            .contiguous()
        )

        return (
            residual + x
        )

In [18]:
class ConditionalLatentUNet3D(nn.Module):

    def __init__(
        self,
        latent_channels=4,
        base_channels=64,
        condition_dim=256,
        entropy_scale=0.1
    ):
        super().__init__()

        self.entropy_scale = (
            entropy_scale
        )

        # ============================
        # Time condition
        # ============================

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(
                condition_dim
            ),
            nn.Linear(
                condition_dim,
                condition_dim
            ),
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                condition_dim
            )
        )

        # ============================
        # Entropy condition
        # ============================

        self.entropy_embedding = nn.Sequential(
            nn.Linear(
                1,
                condition_dim
            ),
            nn.SiLU(),
            nn.Linear(
                condition_dim,
                condition_dim
            )
        )

        nn.init.zeros_(
            self.entropy_embedding[-1].weight
        )

        nn.init.zeros_(
            self.entropy_embedding[-1].bias
        )

        # ============================
        # Latent input
        # ============================

        self.input_conv = nn.Conv3d(
            latent_channels,
            64,
            kernel_size=3,
            padding=1
        )

        # ============================
        # Mask projections
        # ============================

        self.mask_level1 = nn.Conv3d(
            3,
            64,
            kernel_size=3,
            padding=1
        )

        self.mask_level2 = nn.Conv3d(
            3,
            128,
            kernel_size=3,
            padding=1
        )

        self.mask_level3 = nn.Conv3d(
            3,
            256,
            kernel_size=3,
            padding=1
        )

        # Start mask influence softly
        nn.init.zeros_(
            self.mask_level1.weight
        )
        nn.init.zeros_(
            self.mask_level1.bias
        )

        nn.init.zeros_(
            self.mask_level2.weight
        )
        nn.init.zeros_(
            self.mask_level2.bias
        )

        nn.init.zeros_(
            self.mask_level3.weight
        )
        nn.init.zeros_(
            self.mask_level3.bias
        )

        # ============================
        # Encoder level 1
        # 52 x 56 x 40
        # ============================

        self.enc1a = ResBlock3D(
            64,
            64,
            condition_dim
        )

        self.enc1b = ResBlock3D(
            64,
            64,
            condition_dim
        )

        self.down1 = nn.Conv3d(
            64,
            128,
            kernel_size=4,
            stride=2,
            padding=1
        )

        # ============================
        # Encoder level 2
        # 26 x 28 x 20
        # ============================

        self.enc2a = ResBlock3D(
            128,
            128,
            condition_dim
        )

        self.enc2b = ResBlock3D(
            128,
            128,
            condition_dim
        )

        self.down2 = nn.Conv3d(
            128,
            256,
            kernel_size=4,
            stride=2,
            padding=1
        )

        # ============================
        # Bottleneck
        # 13 x 14 x 10
        # ============================

        self.mid1 = ResBlock3D(
            256,
            256,
            condition_dim
        )

        self.mid_attention = (
            SelfAttention3D(
                256,
                heads=8
            )
        )

        self.mid2 = ResBlock3D(
            256,
            256,
            condition_dim
        )

        # ============================
        # Decoder
        # ============================

        self.up2 = nn.ConvTranspose3d(
            256,
            128,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec2a = ResBlock3D(
            256,
            128,
            condition_dim
        )

        self.dec2b = ResBlock3D(
            128,
            128,
            condition_dim
        )

        self.up1 = nn.ConvTranspose3d(
            128,
            64,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec1a = ResBlock3D(
            128,
            64,
            condition_dim
        )

        self.dec1b = ResBlock3D(
            64,
            64,
            condition_dim
        )

        self.out_norm = nn.GroupNorm(
            8,
            64
        )

        self.out_conv = nn.Conv3d(
            64,
            latent_channels,
            kernel_size=3,
            padding=1
        )

        nn.init.zeros_(
            self.out_conv.weight
        )

        nn.init.zeros_(
            self.out_conv.bias
        )


    def forward(
        self,
        z,
        t,
        latent_mask,
        entropy
    ):

        # ============================
        # Global condition
        # ============================

        time_emb = self.time_embedding(
            t
        )

        entropy = (
            entropy
            .float()
            .view(-1, 1)
        )

        entropy_emb = (
            self.entropy_embedding(
                entropy
            )
        )

        condition = (
            time_emb
            +
            self.entropy_scale
            * entropy_emb
        )

        # ============================
        # Level 1 mask
        # ============================

        x = self.input_conv(z)

        mask1 = (
            0.25
            * torch.tanh(
                self.mask_level1(
                    latent_mask
                )
            )
        )

        x = x + mask1

        x = self.enc1a(
            x,
            condition
        )

        x = self.enc1b(
            x,
            condition
        )

        skip1 = x

        # ============================
        # Level 2
        # ============================

        x = self.down1(x)

        latent_mask2 = F.interpolate(
            latent_mask,
            size=x.shape[2:],
            mode="nearest"
        )

        mask2 = (
            0.20
            * torch.tanh(
                self.mask_level2(
                    latent_mask2
                )
            )
        )

        x = x + mask2

        x = self.enc2a(
            x,
            condition
        )

        x = self.enc2b(
            x,
            condition
        )

        skip2 = x

        # ============================
        # Bottleneck
        # ============================

        x = self.down2(x)

        latent_mask3 = F.interpolate(
            latent_mask,
            size=x.shape[2:],
            mode="nearest"
        )

        mask3 = (
            0.15
            * torch.tanh(
                self.mask_level3(
                    latent_mask3
                )
            )
        )

        x = x + mask3

        x = self.mid1(
            x,
            condition
        )

        x = self.mid_attention(x)

        x = self.mid2(
            x,
            condition
        )

        # ============================
        # Decoder level 2
        # ============================

        x = self.up2(x)

        assert (
            x.shape[2:]
            == skip2.shape[2:]
        )

        x = torch.cat(
            [
                x,
                skip2
            ],
            dim=1
        )

        x = self.dec2a(
            x,
            condition
        )

        x = self.dec2b(
            x,
            condition
        )

        # ============================
        # Decoder level 1
        # ============================

        x = self.up1(x)

        assert (
            x.shape[2:]
            == skip1.shape[2:]
        )

        x = torch.cat(
            [
                x,
                skip1
            ],
            dim=1
        )

        x = self.dec1a(
            x,
            condition
        )

        x = self.dec1b(
            x,
            condition
        )

        x = F.silu(
            self.out_norm(x)
        )

        return self.out_conv(x)

In [19]:
class EMA:

    def __init__(
        self,
        model,
        decay=0.9999
    ):
        self.decay = decay


        self.ema_model = copy.deepcopy(
            model
        )


        self.ema_model.eval()


        for parameter in (
            self.ema_model.parameters()
        ):
            parameter.requires_grad = False


def load_ldm_checkpoint(
    model,
    ema,
    path,
    device
):
    checkpoint = torch.load(
        path,
        map_location=device
    )


    required_keys = {
        "epoch",
        "model_state_dict",
        "ema_state_dict",
        "latent_mean",
        "latent_std",
        "entropy_mean",
        "entropy_std"
    }


    missing_keys = (
        required_keys
        - set(checkpoint.keys())
    )


    if missing_keys:
        raise KeyError(
            "LDM checkpoint is missing keys: "
            f"{sorted(missing_keys)}"
        )


    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )


    ema.ema_model.load_state_dict(
        checkpoint[
            "ema_state_dict"
        ]
    )


    latent_mean = (
        torch.as_tensor(
            checkpoint[
                "latent_mean"
            ],
            dtype=torch.float32
        )
        .detach()
        .cpu()
        .view(1, 4, 1, 1, 1)
    )


    latent_std = (
        torch.as_tensor(
            checkpoint[
                "latent_std"
            ],
            dtype=torch.float32
        )
        .detach()
        .cpu()
        .view(1, 4, 1, 1, 1)
    )


    entropy_mean = float(
        torch.as_tensor(
            checkpoint[
                "entropy_mean"
            ]
        ).item()
    )


    entropy_std = float(
        torch.as_tensor(
            checkpoint[
                "entropy_std"
            ]
        ).item()
    )


    if not torch.all(
        latent_std > 0
    ):
        raise RuntimeError(
            "Invalid latent standard deviation "
            "in the LDM checkpoint."
        )


    if entropy_std <= 0:
        raise RuntimeError(
            "Invalid entropy standard deviation "
            f"in the LDM checkpoint: {entropy_std}"
        )


    return (
        int(checkpoint["epoch"]),
        latent_mean,
        latent_std,
        entropy_mean,
        entropy_std
    )

In [20]:
# ============================================================
# Load final Conditional LDM V4 checkpoint
# ============================================================

LDM_CKPT_PATH = (
    "conditional_ldm_v4_checkpoints/"
    "conditional_ldm_v4_epoch_050.pt"
)


if not os.path.isfile(LDM_CKPT_PATH):
    raise FileNotFoundError(
        f"LDM checkpoint not found: {LDM_CKPT_PATH}"
    )


model = ConditionalLatentUNet3D(
    latent_channels=4,
    base_channels=64,
    condition_dim=256,
    entropy_scale=0.1
).to(device)


ema = EMA(
    model,
    decay=0.9999
)


total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


(
    loaded_epoch,
    LATENT_MEAN,
    LATENT_STD,
    ENTROPY_MEAN,
    ENTROPY_STD
) = load_ldm_checkpoint(
    model=model,
    ema=ema,
    path=LDM_CKPT_PATH,
    device=device
)


if loaded_epoch != 50:
    raise RuntimeError(
        "Expected LDM epoch 50, "
        f"but loaded epoch {loaded_epoch}."
    )


sampling_model = ema.ema_model
sampling_model.eval()


# Remove the non-EMA duplicate
del model


if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    "Loaded Conditional LDM V4 epoch:",
    loaded_epoch
)

print(
    "LDM checkpoint:",
    LDM_CKPT_PATH
)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "LATENT_MEAN:",
    LATENT_MEAN.flatten()
)

print(
    "LATENT_STD:",
    LATENT_STD.flatten()
)

print(
    "Entropy mean:",
    ENTROPY_MEAN
)

print(
    "Entropy std:",
    ENTROPY_STD
)

print(
    "Sampling model: EMA"
)

Loaded Conditional LDM V4 epoch: 50
LDM checkpoint: conditional_ldm_v4_checkpoints/conditional_ldm_v4_epoch_050.pt
Total parameters: 18,517,444
LATENT_MEAN: tensor([-0.0313, -0.1939,  0.1352,  0.0145])
LATENT_STD: tensor([0.6804, 0.9484, 1.4222, 0.2118])
Entropy mean: 6.870205879211426
Entropy std: 0.33203670382499695
Sampling model: EMA


In [21]:
@torch.no_grad()
def sample_conditional_ldm(
    model,
    vae,
    mask,
    entropy,
    device,
    seed=42
):

    model.eval()
    vae.eval()

    torch.manual_seed(seed)

    latent_mean = (
        LATENT_MEAN
        .to(device)
    )

    latent_std = (
        LATENT_STD
        .to(device)
    )

    latent_mask = (
        prepare_latent_mask(
            mask.to(device)
        )
    )

    entropy = (
        entropy
        .to(device)
        .float()
    )

    entropy = (
        entropy
        - ENTROPY_MEAN
    ) / ENTROPY_STD


    x = torch.randn(
        (
            mask.shape[0],
            4,
            52,
            56,
            40
        ),
        device=device
    )


    coef1 = (
        posterior_mean_coef1
        .to(device)
    )

    coef2 = (
        posterior_mean_coef2
        .to(device)
    )

    posterior_var = (
        posterior_variance
        .to(device)
    )


    for step in reversed(
        range(timesteps)
    ):

        t = torch.full(
            (x.shape[0],),
            step,
            device=device,
            dtype=torch.long
        )

        v_pred = model(
            x,
            t,
            latent_mask,
            entropy
        )

        x0_pred = v_to_x0(
            x,
            v_pred,
            t
        )

        # Avoid exploding latent prediction
        x0_pred = torch.clamp(
            x0_pred,
            -5.0,
            5.0
        )

        model_mean = (
            coef1[step]
            * x0_pred
            +
            coef2[step]
            * x
        )

        if step > 0:

            noise = torch.randn_like(x)

            x = (
                model_mean
                +
                torch.sqrt(
                    posterior_var[step]
                )
                * noise
            )

        else:

            x = model_mean


    # Return to original VAE latent distribution
    generated_latent = (
        x * latent_std
        + latent_mean
    )

    generated_image = (
        vae.decoder(
            generated_latent
        )
    )

    return (
        generated_image,
        generated_latent
    )

In [22]:
# ============================================================
# Conditional LDM V4 generation configuration
# ============================================================

NUM_TO_GENERATE = 200


BASE_SEED = 30000


METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "metadata_conditional_ldm_v4.csv"
)


AFFINE = np.eye(
    4,
    dtype=np.float32
)


if NUM_TO_GENERATE < 1:
    raise ValueError(
        "NUM_TO_GENERATE must be at least 1."
    )


if NUM_TO_GENERATE > len(
    evaluation_dataset
):
    raise ValueError(
        "NUM_TO_GENERATE cannot exceed "
        f"the cohort size "
        f"{len(evaluation_dataset)}."
    )


print(
    "Number to generate:",
    NUM_TO_GENERATE
)

print(
    "Available condition subjects:",
    len(evaluation_dataset)
)

print(
    "Synthetic output directory:",
    OUTPUT_DIR
)

print(
    "Shared condition directory:",
    CONDITION_DIR
)

print(
    "Metadata:",
    METADATA_PATH
)

print(
    "Seed range:",
    BASE_SEED,
    "to",
    BASE_SEED + NUM_TO_GENERATE - 1
)

Number to generate: 200
Available condition subjects: 200
Synthetic output directory: evaluation_200/conditional_ldm_v4
Shared condition directory: evaluation_200/conditions
Metadata: evaluation_200/conditional_ldm_v4/metadata_conditional_ldm_v4.csv
Seed range: 30000 to 30199


In [23]:
# ============================================================
# Generate Conditional LDM V4 evaluation volumes
# ============================================================

metadata_exists = os.path.isfile(
    METADATA_PATH
)


existing_metadata_ids = set()


if metadata_exists:

    with open(
        METADATA_PATH,
        "r",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:
            existing_metadata_ids.add(
                row["sample_id"]
            )


else:

    with open(
        METADATA_PATH,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "sample_id",
            "model",
            "source_subject",
            "filename",
            "condition_mask_filename",
            "seed",
            "raw_entropy",
            "z_entropy",
            "shape_x",
            "shape_y",
            "shape_z",
            "min",
            "max",
            "mean",
            "std",
            "generation_seconds",
            "status"
        ])


def append_metadata(
    sample_id,
    subject,
    filename,
    mask_filename,
    seed,
    raw_entropy,
    z_entropy,
    volume,
    generation_seconds,
    status
):

    with open(
        METADATA_PATH,
        "a",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            sample_id,
            "conditional_ldm_v4",
            subject,
            filename,
            mask_filename,
            seed,
            raw_entropy,
            z_entropy,
            volume.shape[0],
            volume.shape[1],
            volume.shape[2],
            float(volume.min()),
            float(volume.max()),
            float(volume.mean()),
            float(volume.std()),
            generation_seconds,
            status
        ])


total_start = time.perf_counter()

generated_this_run = 0


for i in range(
    NUM_TO_GENERATE
):

    sample_id = f"{i:04d}"

    seed = (
        BASE_SEED
        + i
    )


    sample = evaluation_dataset[i]


    subject = sample[
        "subject"
    ]


    raw_entropy = float(
        sample[
            "heterogeneity"
        ].item()
    )


    z_entropy = (
        raw_entropy
        - ENTROPY_MEAN
    ) / ENTROPY_STD


    filename = (
        f"conditional_ldm_v4_"
        f"{sample_id}.nii.gz"
    )


    output_path = os.path.join(
        OUTPUT_DIR,
        filename
    )


    mask_filename = (
        f"condition_mask_"
        f"{sample_id}.nii.gz"
    )


    mask_path = os.path.join(
        MASK_DIR,
        mask_filename
    )


    # --------------------------------------------------------
    # Resume support
    # --------------------------------------------------------

    if os.path.isfile(
        output_path
    ):

        print(
            f"[{i + 1:03d}/{NUM_TO_GENERATE}] "
            f"{filename} already exists -> skipped"
        )


        if sample_id not in existing_metadata_ids:

            existing_volume = np.asarray(
                nib.load(
                    output_path
                ).dataobj,
                dtype=np.float32
            )


            append_metadata(
                sample_id=sample_id,
                subject=subject,
                filename=filename,
                mask_filename=mask_filename,
                seed=seed,
                raw_entropy=raw_entropy,
                z_entropy=z_entropy,
                volume=existing_volume,
                generation_seconds="",
                status="recovered_existing"
            )


            existing_metadata_ids.add(
                sample_id
            )


            del existing_volume


        del sample

        continue


    if sample_id in existing_metadata_ids:

        print(
            "Warning: metadata existed without "
            f"volume for sample {sample_id}; "
            "a new row will be written."
        )

        existing_metadata_ids.remove(
            sample_id
        )


    print()

    print(
        f"[{i + 1:03d}/{NUM_TO_GENERATE}] "
        f"Generating {filename}"
    )

    print(
        "Source subject:",
        subject
    )

    print(
        "Seed:",
        seed
    )

    print(
        "Raw entropy:",
        raw_entropy
    )

    print(
        "Z entropy:",
        z_entropy
    )


    torch.manual_seed(
        seed
    )

    np.random.seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    mask_batch = (
        sample["mask"]
        .unsqueeze(0)
    )


    entropy_batch = (
        sample["heterogeneity"]
        .unsqueeze(0)
    )


    sample_start = time.perf_counter()


    (
        generated_image,
        generated_latent
    ) = sample_conditional_ldm(
        model=sampling_model,
        vae=vae,
        mask=mask_batch,
        entropy=entropy_batch,
        device=device,
        seed=seed
    )


    if device.type == "cuda":
        torch.cuda.synchronize()


    sample_seconds = (
        time.perf_counter()
        - sample_start
    )


    volume = (
        generated_image[
            0,
            0
        ]
        .detach()
        .float()
        .cpu()
        .numpy()
    )


    # The VAE decoder already uses sigmoid,
    # so the generated image is already [0,1].
    volume = np.clip(
        volume,
        0.0,
        1.0
    ).astype(
        np.float32
    )


    expected_shape = (
        208,
        224,
        160
    )


    if volume.shape != expected_shape:

        raise RuntimeError(
            "Unexpected generated shape: "
            f"{volume.shape}"
        )


    if not np.all(
        np.isfinite(volume)
    ):

        raise RuntimeError(
            "NaN or Inf found in "
            f"sample {sample_id}"
        )


    synthetic_nifti = nib.Nifti1Image(
        volume,
        AFFINE
    )


    synthetic_nifti.set_data_dtype(
        np.float32
    )


    nib.save(
        synthetic_nifti,
        output_path
    )


    append_metadata(
        sample_id=sample_id,
        subject=subject,
        filename=filename,
        mask_filename=mask_filename,
        seed=seed,
        raw_entropy=raw_entropy,
        z_entropy=z_entropy,
        volume=volume,
        generation_seconds=sample_seconds,
        status="generated"
    )


    existing_metadata_ids.add(
        sample_id
    )


    generated_this_run += 1


    completed_files = len([
        name
        for name in os.listdir(
            OUTPUT_DIR
        )
        if (
            name.startswith(
                "conditional_ldm_v4_"
            )
            and name.endswith(
                ".nii.gz"
            )
        )
    ])


    completed_files = min(
        completed_files,
        NUM_TO_GENERATE
    )


    remaining = (
        NUM_TO_GENERATE
        - completed_files
    )


    estimated_remaining_hours = (
        remaining
        * sample_seconds
        / 3600.0
    )


    print(
        "Saved:",
        output_path
    )


    if os.path.isfile(
        mask_path
    ):

        print(
            "Shared condition mask:",
            mask_path
        )

    else:

        print(
            "Shared condition mask not yet present:",
            mask_path
        )


    print(
        "Shape:",
        volume.shape
    )

    print(
        "Range:",
        float(volume.min()),
        float(volume.max())
    )

    print(
        "Mean:",
        float(volume.mean())
    )

    print(
        "Std:",
        float(volume.std())
    )

    print(
        f"Generation time: "
        f"{sample_seconds / 60:.2f} min"
    )

    print(
        f"Completed: "
        f"{completed_files}/"
        f"{NUM_TO_GENERATE}"
    )

    print(
        f"Estimated remaining time: "
        f"{estimated_remaining_hours:.2f} h"
    )


    del sample
    del mask_batch
    del entropy_batch
    del generated_image
    del generated_latent
    del volume
    del synthetic_nifti


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


total_seconds = (
    time.perf_counter()
    - total_start
)


print()

print(
    "========================================"
)

print(
    "Conditional LDM V4 generation finished"
)

print(
    "========================================"
)

print(
    "Generated during this run:",
    generated_this_run
)

print(
    f"Runtime this session: "
    f"{total_seconds / 3600:.2f} h"
)

print(
    "Synthetic output directory:",
    OUTPUT_DIR
)

print(
    "Shared condition directory:",
    CONDITION_DIR
)

print(
    "Metadata:",
    METADATA_PATH
)

[001/200] conditional_ldm_v4_0000.nii.gz already exists -> skipped



[002/200] Generating conditional_ldm_v4_0001.nii.gz
Source subject: BraTS-GLI-00756-000
Seed: 30001
Raw entropy: 7.1754231452941895
Z entropy: 0.9192274907163013


Saved: evaluation_200/conditional_ldm_v4/conditional_ldm_v4_0001.nii.gz
Shared condition mask: evaluation_200/conditions/masks/condition_mask_0001.nii.gz
Shape: (208, 224, 160)
Range: 2.7144933436445973e-12 0.9724207520484924
Mean: 0.08320745080709457
Std: 0.17409658432006836
Generation time: 0.54 min
Completed: 2/200
Estimated remaining time: 1.77 h



[003/200] Generating conditional_ldm_v4_0002.nii.gz
Source subject: BraTS-GLI-01215-000
Seed: 30002
Raw entropy: 6.929394245147705
Z entropy: 0.17825850351615066


Saved: evaluation_200/conditional_ldm_v4/conditional_ldm_v4_0002.nii.gz
Shared condition mask not yet present: evaluation_200/conditions/masks/condition_mask_0002.nii.gz
Shape: (208, 224, 160)
Range: 4.243354365801588e-12 0.9840699434280396
Mean: 0.08507363498210907
Std: 0.17881610989570618
Generation time: 0.53 min
Completed: 3/200
Estimated remaining time: 1.74 h


...

Saved: evaluation_200/conditional_ldm_v4/conditional_ldm_v4_0197.nii.gz
Shared condition mask not yet present: evaluation_200/conditions/masks/condition_mask_0197.nii.gz
Shape: (208, 224, 160)
Range: 3.25311583226906e-11 0.9748674035072327
Mean: 0.08090011775493622
Std: 0.1733531355857849
Generation time: 0.53 min
Completed: 198/200
Estimated remaining time: 0.02 h



[199/200] Generating conditional_ldm_v4_0198.nii.gz
Source subject: BraTS-GLI-01368-000
Seed: 30198
Raw entropy: 6.876424789428711
Z entropy: 0.018729586656067068


Saved: evaluation_200/conditional_ldm_v4/conditional_ldm_v4_0198.nii.gz
Shared condition mask not yet present: evaluation_200/conditions/masks/condition_mask_0198.nii.gz
Shape: (208, 224, 160)
Range: 4.205644946880804e-12 0.9630247354507446
Mean: 0.08053905516862869
Std: 0.178122416138649
Generation time: 0.53 min
Completed: 199/200
Estimated remaining time: 0.01 h



[200/200] Generating conditional_ldm_v4_0199.nii.gz
Source subject: BraTS-GLI-00621-000
Seed: 30199
Raw entropy: 6.574536323547363
Z entropy: -0.8904725057742349


Saved: evaluation_200/conditional_ldm_v4/conditional_ldm_v4_0199.nii.gz
Shared condition mask not yet present: evaluation_200/conditions/masks/condition_mask_0199.nii.gz
Shape: (208, 224, 160)
Range: 1.7343438358098477e-14 0.9543817639350891
Mean: 0.0978873148560524
Std: 0.20539475977420807
Generation time: 0.53 min
Completed: 200/200
Estimated remaining time: 0.00 h

Conditional LDM V4 generation finished
Generated during this run: 199
Runtime this session: 1.83 h
Synthetic output directory: evaluation_200/conditional_ldm_v4
Shared condition directory: evaluation_200/conditions
Metadata: evaluation_200/conditional_ldm_v4/metadata_conditional_ldm_v4.csv
